# Fig 5 — Axolotl regeneration trajectories (HEIST)

Renders the MIOFlow Fig 5 for a given run `SUFFIX`. Default `_l0p25` = gene-PHATE + 0.25·HEIST-spatial
(the best latent). Artifacts are read from `outputs/{h5ad,trajectories}` via `paths.py`.
Run this notebook from the `mioflow_heist/` directory.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

sys.path.insert(0, os.getcwd())
import paths

# Default = best figure: per-section (batch-corrected at preprocessing) HEIST embedding,
# gene-PHATE + 0.5*HEIST-spatial, knn=7/t=15/GAGA. Base = axolotl_heist_persec.h5ad.
# Other runs: '_l0p5' (global, batch-split), '_l0p5_hz' (latent Harmony), '_l0p5_ps_direct'.
SUFFIX = os.environ.get('HEIST_SUFFIX', '_l0p5_ps')
LR = 72                                              # NCAN:SDC3
GOIS = [6009, 11395, 8976, 4243]
GNAMES = ['VIM', 'SLC1A3', 'STMN4', 'S100A10']
EMB_KEY = 'X_gaga_2stage_heist'
CT_KEY = 'Annotation_grouped'
print('plotting SUFFIX =', SUFFIX)

In [ ]:
def llcenter(X):
    return X - np.min(X, axis=0)
def notick(ax):
    ax.set_xticks([]); ax.set_yticks([])
def gradline(ax, trajs, c, cmap='magma', alpha=1, lw=1):
    x0s = trajs[:, 0:-1, 0].reshape(-1, 1); x1s = trajs[:, 1:, 0].reshape(-1, 1)
    y0s = trajs[:, 0:-1, 1].reshape(-1, 1); y1s = trajs[:, 1:, 1].reshape(-1, 1)
    pts = np.stack((np.concatenate((x0s, y0s), 1), np.concatenate((x1s, y1s), 1)), axis=1)
    colors = ((c[..., 1:] + c[..., 0:-1]) / 2).flatten()
    lc = LineCollection(pts, array=colors, cmap=cmap, alpha=alpha, lw=lw, capstyle='round')
    ax.add_collection(lc); return lc

In [ ]:
data = sc.read_h5ad(paths.h5ad(f'axolotl_heist{SUFFIX}.h5ad'))
data.obs_names_make_unique()
ptrajs = np.load(paths.traj(f'ptrajs{SUFFIX}.npy'))[:100]
gtrajs = np.load(paths.traj(f'gtrajs{SUFFIX}.npy'))[:100]
strajs = np.load(paths.traj(f'strajs{SUFFIX}.npy'))[:100]
emb = np.asarray(data.obsm[EMB_KEY])
lr_static = np.asarray(data.obsm['LR_feats'])[:, LR]
tb = np.asarray(data.obs['time_bin'], dtype=float)
tx = np.linspace(tb.min(), tb.max(), ptrajs.shape[1])
cts = list(np.unique(data.obs[CT_KEY]))
print('ptrajs', ptrajs.shape, 'gtrajs', gtrajs.shape, 'strajs', strajs.shape)

In [ ]:
smean = strajs[..., LR].mean(0); sdev = strajs[..., LR].std(0)

# 3-row layout: row1 = 3 embedding scatters + trajectories; row2 = 3 DPI panels + both
# NCAN:SDC3 curves; row3 = gene trends. 20-col grid (row1/3 span 5, row2 spans 4).
fig = plt.figure(figsize=(28, 18))
gs = fig.add_gridspec(3, 20, hspace=0.32, wspace=2.2)
C4 = [(0, 5), (5, 10), (10, 15), (15, 20)]
C5 = [(0, 4), (4, 8), (8, 12), (12, 16), (16, 20)]
def cell(r, span): return fig.add_subplot(gs[r, span[0]:span[1]])

# Row 1: embedding (PHATE) scatters + trajectories
ax = cell(0, C4[0])
for ct in cts:
    m = (data.obs[CT_KEY] == ct).to_numpy()
    ax.scatter(*emb[m].T, s=3, label=ct)
ax.legend(markerscale=4, title='Cell Type', fontsize=8)
ax.set_title('HEIST Embedding — Cell Type'); notick(ax)

ax = cell(0, C4[1])
sctp = ax.scatter(*emb.T, s=3, c=tb, cmap='coolwarm', vmin=0, vmax=1)
fig.colorbar(sctp, ax=ax, fraction=0.046); ax.set_title('HEIST Embedding — Pseudotime'); notick(ax)

ax = cell(0, C4[2])
scls = ax.scatter(*emb.T, s=3, c=lr_static, cmap='magma', vmin=0)
fig.colorbar(scls, ax=ax, fraction=0.046); ax.set_title('HEIST Embedding — NCAN:SDC3'); notick(ax)

ax = cell(0, C4[3])
ax.scatter(*emb.T, s=3, c='lightgrey')
lc = gradline(ax, ptrajs[:, :, 0:2], strajs[..., LR], cmap='magma', lw=2)
fig.colorbar(lc, ax=ax, fraction=0.046)
ax.set_title('Trajectories (NCAN:SDC3 signaling)')
ax.set_xlabel('HEIST-latent 1'); ax.set_ylabel('HEIST-latent 2'); ax.set_box_aspect(1)

# Row 2: spatial DPI panels + both NCAN:SDC3 curves
try:
    alldata = sc.read_h5ad(paths.AXOLOTL_FULL, backed='r')
    obs = alldata.obs
    sp_all = np.asarray(alldata.obsm['spatial'])
    lr_all = np.asarray(alldata.obsm['LR_feats'])[:, LR]
    inj = (obs['inj_uninj'].astype(str) == 'inj').to_numpy()
    dpi = obs['dpi'].astype(str).to_numpy()
    batch = obs['Batch'].astype(str).to_numpy()
    vmax_sp = float(np.max(np.asarray(data.obsm['LR_feats'])[:, LR]))
    sp_plot = None
    for i, t in enumerate([2, 15, 60]):
        ax = cell(1, C5[i])
        m = inj & (dpi == str(t))
        if m.sum() == 0:
            ax.set_title(f'{t} DPI (no cells)'); notick(ax); continue
        m = m & (batch == np.unique(batch[m])[0])
        sp_plot = ax.scatter(*(llcenter(sp_all[m]) / 2).T, s=4, c=lr_all[m], cmap='magma', vmin=0, vmax=vmax_sp)
        ax.set_title(f'{t} DPI'); ax.set_aspect('equal'); notick(ax)
    if sp_plot is not None:
        fig.colorbar(sp_plot, ax=ax, fraction=0.046)
except Exception as e:
    print('spatial DPI panels skipped:', type(e).__name__, e)

ax = cell(1, C5[3])
ax.plot(tx, smean, lw=3, c='k'); ax.fill_between(tx, smean - sdev, smean + sdev, color='grey', alpha=0.3)
ax.set_xlabel('Pseudotime'); ax.set_ylabel('Signaling Level'); ax.set_title('NCAN:SDC3 Signaling')

ax = cell(1, C5[4])
ax.plot(tx, smean, lw=3, c='k'); ax.fill_between(tx, smean - sdev, smean + sdev, color='grey', alpha=0.3)
ax.set_xlabel('Pseudotime'); ax.set_ylabel('Signaling'); ax.set_title('NCAN:SDC3 over Pseudotime')

# Row 3: gene trends
for j, (g, nm) in enumerate(zip(GOIS, GNAMES)):
    ax = cell(2, C4[j])
    gmean = gtrajs[..., g].mean(0); gstd = gtrajs[..., g].std(0)
    ax.plot(tx, gmean, c='k'); ax.fill_between(tx, gmean - gstd, gmean + gstd, color='grey', alpha=0.3)
    ax.set_title(nm); ax.set_xlabel('Pseudotime')
    if j == 0:
        ax.set_ylabel('Expression Level')

fig.suptitle(f'Fig 5 (HEIST) — Axolotl regeneration trajectories [{SUFFIX}]', fontsize=16, y=0.995)
fig.savefig(paths.fig(f'fig5_heist{SUFFIX}.png'), dpi=130, bbox_inches='tight')
plt.show()